# 02 · Download race laps, weather and results (2018–2025)

Saves the **raw** data for every race to Google Drive: every lap of every driver, session weather (about one reading per minute) and the race results. Nothing is filtered yet; which laps count as "clean" is decided later in the analysis.

**This takes several runs.** Each race costs roughly ten API calls and FastF1 stops at its hourly limits, so 2018–2025 needs about four hours of reruns. When a run stops at the limit, wait about an hour and run the download cell again; finished races are skipped.

**Drive space:** FastF1's cache of raw API responses grows with each race. The last cell shows how much space it's using.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Download (rerun this cell until nothing is left)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}
QUIET_LOGS = True  # @param {type:"boolean"}

from f1sim.paths import get_paths
from f1sim.ingest import quiet_fastf1_logs
from f1sim.ingest_race import ingest_race_seasons

quiet_fastf1_logs(QUIET_LOGS)
paths = get_paths()
summary = ingest_race_seasons(range(START_YEAR, END_YEAR + 1), paths)

In [ ]:
# @title Coverage: races, laps and rain per season
from f1sim.ingest_race import load_race

laps = load_race("laps", paths)
weather = load_race("weather", paths)

races = (
    laps.groupby(["Year", "Round", "Event"])
    .agg(Laps=("LapNumber", "size"), Drivers=("Driver", "nunique"))
    .reset_index()
)
rain = weather.groupby(["Year", "Round"])["Rainfall"].any().rename("AnyRain")
races = races.join(rain, on=["Year", "Round"])
races["AnyRain"] = races["AnyRain"].astype("boolean")

races.groupby("Year").agg(
    Races=("Round", "size"),
    Laps=("Laps", "sum"),
    RacesWithRain=("AnyRain", "sum"),
    WeatherMissing=("AnyRain", lambda s: int(s.isna().sum())),
)

In [ ]:
# @title Races where rain was recorded at any point
races[races["AnyRain"] == True][["Year", "Round", "Event", "Laps"]]

In [ ]:
# @title Drive space used
!du -sh "{paths.cache_dir}" "{paths.race_dir}" "{paths.quali_dir}"